In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [0]:
API_KEY = dbutils.secrets.get(scope="tc_02", key="api_key")
API_SECRET = dbutils.secrets.get(scope="tc_02", key="api_secret")
BOOTSTRAP_SERVER = dbutils.secrets.get(scope="tc_02", key="bootstrap_servers")

bootstrap = BOOTSTRAP_SERVER if ":" in BOOTSTRAP_SERVER else f"{BOOTSTRAP_SERVER}:9092"

In [0]:
FINAL_SCHEMA = 'origens'
FINAL_TABLE  = 'tc02_uf'

In [0]:
schema = StructType([
    StructField("ano", IntegerType(), True),
    StructField("sigla_uf", StringType(), True),
    StructField("serie", StringType(), True),
    StructField("rede", StringType(), True),
    StructField("taxa_alfabetizacao", DoubleType(), True),
    StructField("media_portugues", DoubleType(), True),
    StructField("percentual_participacao", DoubleType(), True),
    StructField("proporcao_aluno_nivel_0", DoubleType(), True),
    StructField("proporcao_aluno_nivel_1", DoubleType(), True),
    StructField("proporcao_aluno_nivel_2", DoubleType(), True),
    StructField("proporcao_aluno_nivel_3", DoubleType(), True),
    StructField("proporcao_aluno_nivel_4", DoubleType(), True),
    StructField("proporcao_aluno_nivel_5", DoubleType(), True),
    StructField("proporcao_aluno_nivel_6", DoubleType(), True),
    StructField("proporcao_aluno_nivel_7", DoubleType(), True),
    StructField("proporcao_aluno_nivel_8", DoubleType(), True),
])

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FINAL_SCHEMA}")

In [0]:
kafka_options = {
    "kafka.bootstrap.servers": bootstrap,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="{API_KEY}" password="{API_SECRET}";'
    ),
    "subscribe": "uf-eventos",
    "startingOffsets": "earliest",
}

df = spark.readStream.format("kafka").options(**kafka_options).load()
df_parsed = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)", "timestamp")

df_parsed = df_parsed.withColumn("value_parsed", F.from_json(F.col("value"), schema))

df_final = df_parsed.select("value_parsed.*")

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.origens.checkpoints")

checkpoint_path = f"/Volumes/workspace/origens/checkpoints/tc02_uf_streaming"

query = (
    df_final
    .withColumn("_data_criacao_origem", F.current_timestamp())
    .withColumn("_ano_ingestao", F.year(F.current_timestamp()).cast("short"))
    .withColumn("_mes_ingestao", F.month(F.current_timestamp()).cast("byte"))
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .partitionBy("_ano_ingestao", "_mes_ingestao")
    .trigger(availableNow=True)
    .toTable(f"{FINAL_SCHEMA}.{FINAL_TABLE}")
)

query.processAllAvailable()
print("Lote processado. Status final:", query.status)

In [0]:
# spark.sql(f"SELECT * FROM {FINAL_SCHEMA}.{FINAL_TABLE} WHERE sigla_uf = 'TEST' LIMIT 10").display()

In [0]:
# spark.sql(f"DELETE FROM {FINAL_SCHEMA}.{FINAL_TABLE} WHERE sigla_uf = 'TEST'")
